In [5]:
# ---- enrich_jsonl_in_place.py ----
# Run inside a notebook cell or as `python enrich_jsonl_in_place.py`

import json, os, re, sys, tempfile

SRC = "datasets_tasks_full_enriched.jsonl"  # <-- change if needed

def split_owner_title(dataset_id: str):
    if not isinstance(dataset_id, str):
        return "", ""
    if "/" in dataset_id:
        owner, rest = dataset_id.split("/", 1)
        return owner.strip(), rest.strip()
    return "", dataset_id.strip()

def first_match(urls, pred):
    if not isinstance(urls, list):
        return ""
    for u in urls:
        if isinstance(u, str) and pred(u):
            return u
    return ""

def is_github(u: str) -> bool:
    return isinstance(u, str) and "github.com" in u

def normalize_arxiv(u: str) -> str:
    if not isinstance(u, str) or "arxiv.org" not in u:
        return ""
    if "/abs/" in u:
        return u
    m = re.search(r"arxiv\.org/(?:pdf|format)/([^/\s]+)", u)
    return f"https://arxiv.org/abs/{m.group(1)}" if m else u

def best_paper_link(rec: dict) -> str:
    # 1) explicit paper url
    p = rec.get("paper")
    if isinstance(p, str) and p.startswith(("http://", "https://")):
        return normalize_arxiv(p) or p
    # 2) arxiv ids
    ids = rec.get("arxiv_ids_clean") or []
    if isinstance(ids, list) and ids:
        return f"https://arxiv.org/abs/{ids[0]}"
    # 3) arxiv in source_urls
    arx = first_match(rec.get("source_urls"), lambda u: "arxiv.org" in u)
    return normalize_arxiv(arx)

def best_github_link(rec: dict) -> str:
    urls = []
    for key in ("homepage", "paper"):
        v = rec.get(key)
        if isinstance(v, str):
            urls.append(v)
    su = rec.get("source_urls")
    if isinstance(su, list):
        urls.extend([u for u in su if isinstance(u, str)])
    return first_match(urls, is_github)

def main(path: str):
    if not os.path.exists(path):
        print(f"File not found: {path}", file=sys.stderr)
        sys.exit(1)

    tmp = path + ".tmp"
    n_in = n_out = 0

    with open(path, "r", encoding="utf-8") as fin, open(tmp, "w", encoding="utf-8") as fout:
        for line in fin:
            if not line.strip():
                continue
            n_in += 1
            try:
                rec = json.loads(line)
            except Exception:
                continue

            dsid = rec.get("dataset_id") or rec.get("id") or ""
            user_id, title_part = split_owner_title(dsid)

            # ALWAYS set these two per your rule
            rec["title"] = title_part
            rec["user_id"] = user_id

            # Links (fill only if missing)
            gh = best_github_link(rec)
            pv = best_paper_link(rec)
            if gh and not rec.get("github"):
                rec["github"] = gh
            if pv and not rec.get("paper"):
                rec["paper"] = pv

            fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
            n_out += 1

    # atomic replace
    os.replace(tmp, path)
    print(f"Updated records: {n_out}  |  File: {path}")

if __name__ == "__main__":
    # Allow overriding path via argv
    main(sys.argv[1] if len(sys.argv) > 1 else SRC)


File not found: --f=c:\Users\annasokol\AppData\Roaming\jupyter\runtime\kernel-v388b5b6e1a293b712f18bacc16f8cf957ea7ea92f.json


SystemExit: 1

c:\ProgramData\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [4]:
%pip install caas_jupyter_tools

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement caas_jupyter_tools (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for caas_jupyter_tools
